# PaperStreet Research

Interactive research notebook for exploring market data and strategy logic.

**Before running:** ensure TWS is open and connected on port 7497.

**Workflow:** run the Setup cell once per session. All other cells can be re-run independently without reconnecting.

**Cell order:**
Fetch → Indicator Analysis → Feature Engineering → ML → Parameter Optimization → best_params → Signal Overlay → Out-of-Sample Validation → Multi-Symbol Comparison → Teardown

## 1. Setup
Connect to TWS once. Re-running this cell will reconnect if the session was closed.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.gridspec as gridspec
import matplotlib.ticker as mticker

from research.session import Session

# Disconnect any existing session before creating a new one
try:
    session.disconnect()
except NameError:
    pass

session = Session()
print("Session ready. TWS connected:", session.is_connected)


def make_x_ticks(index, max_ticks=10):
    """
    Build integer x-axis tick positions and labels from a DatetimeIndex.

    Plots against integer positions (0, 1, 2, ...) rather than timestamps so
    matplotlib never draws connecting lines across overnight or weekend gaps.
    Tick marks are placed at session-open bars and spaced so no more than
    max_ticks labels appear regardless of how many bars are in the data.

    Parameters
    ----------
    index : DatetimeIndex
        The datetime index of the DataFrame being plotted.
    max_ticks : int
        Upper bound on the number of x-axis labels to show.

    Returns
    -------
    tick_positions : list[int]
        Integer positions where ticks should be placed.
    tick_labels : list[str]
        Formatted label for each tick position.
    """
    # A session open is any bar whose date differs from the previous bar's date
    dates = pd.Series(index.date)
    session_opens = [0] + list(dates[dates != dates.shift()].index[1:])

    # Subsample so we never show more than max_ticks labels
    step = max(1, len(session_opens) // max_ticks)
    tick_positions = session_opens[::step]
    tick_labels = [index[i].strftime("%m/%d") for i in tick_positions]

    return tick_positions, tick_labels

## 2. Fetch Data

Pull bars for a symbol. Adjust `SYMBOL`, `BAR_SIZE`, and `DURATION` to explore different instruments and resolutions.

**Bar size / max duration combinations (IBKR enforced):**

| BAR_SIZE   | Max DURATION |
|------------|--------------|
| `1 min`    | `1 W`        |
| `5 mins`   | `1 M`        |
| `15 mins`  | `1 M`        |
| `30 mins`  | `1 M`        |
| `1 hour`   | `1 Y`        |
| `1 day`    | `5 Y`        |

In [ ]:
SYMBOL   = "SPY"
DURATION = "1 M"   # 1 W | 1 M | 3 M | 6 M | 1 Y | 2 Y | 5 Y
BAR_SIZE = "1 min" # 1 min | 5 mins | 15 mins | 30 mins | 1 hour | 1 day

df = session.market_data.get_bars(SYMBOL, duration=DURATION, bar_size=BAR_SIZE)

print(f"\n{SYMBOL} — {len(df)} bars ({DURATION})")
print(f"From : {df.index[0].date()}")
print(f"To   : {df.index[-1].date()}")
print(f"\nShape   : {df.shape}")
print(f"Columns : {list(df.columns)}")
print(f"Dtypes  :\n{df.dtypes}")
df.head()

## 3. Price & Volume Chart
OHLC bar chart with volume subplot.

In [ ]:
x = np.arange(len(df))
tick_pos, tick_labels = make_x_ticks(df.index)

fig = plt.figure(figsize=(14, 7))
gs  = gridspec.GridSpec(2, 1, height_ratios=[3, 1], hspace=0.08)

ax_price  = fig.add_subplot(gs[0])
ax_volume = fig.add_subplot(gs[1], sharex=ax_price)

# --- Price: close line + high/low range shaded ---
ax_price.plot(x, df["close"], color="#1f77b4", linewidth=1.0, label="Close")
ax_price.fill_between(x, df["low"], df["high"], alpha=0.15, color="#1f77b4", label="High/Low range")
ax_price.set_ylabel("Price (USD)")
ax_price.set_title(f"{SYMBOL} — {BAR_SIZE} Price & Volume ({DURATION})")
ax_price.legend(loc="upper left")
ax_price.grid(True, alpha=0.3)
plt.setp(ax_price.get_xticklabels(), visible=False)

# --- Volume ---
ax_volume.bar(x, df["volume"], color="#aec7e8", width=0.8)
ax_volume.set_ylabel("Volume")
ax_volume.grid(True, alpha=0.3)
ax_volume.set_xticks(tick_pos)
ax_volume.set_xticklabels(tick_labels, rotation=45, ha="right")

plt.tight_layout()
plt.show()

## 4. Indicator Analysis — Mean Reversion Strategy Parameters

Computes the same indicators `MeanReversionStrategy` uses internally:
- **SMA** (fair value)
- **Volatility bands** (SMA ± spread_multiplier × rolling std dev)

**Tunable parameters — adjust these and re-run to explore:**

| Parameter          | Description                                                        |
|--------------------|--------------------------------------------------------------------|
| `WINDOW`           | Lookback bars for SMA and volatility. Larger = smoother/slower.   |
| `SPREAD_MULTIPLIER`| How many std devs price must move to trigger a signal.            |
| `MAX_POSITION`     | Hard cap on shares held at once.                                  |
| `ORDER_SIZE`       | Shares per trade.                                                  |

In [ ]:
# --- Strategy parameters: adjust these ---
WINDOW            = 20
SPREAD_MULTIPLIER = 1.0
MAX_POSITION      = 50
ORDER_SIZE        = 10

# --- Compute indicators (mirrors MeanReversionStrategy internals) ---
df["sma"]        = df["close"].rolling(WINDOW).mean()
df["volatility"] = df["close"].rolling(WINDOW).std()
df["upper_band"] = df["sma"] + SPREAD_MULTIPLIER * df["volatility"]
df["lower_band"] = df["sma"] - SPREAD_MULTIPLIER * df["volatility"]
df["deviation"]  = df["close"] - df["sma"]

print(f"Window: {WINDOW} bars | Spread multiplier: {SPREAD_MULTIPLIER} | Max position: {MAX_POSITION} | Order size: {ORDER_SIZE}")
print(f"\nCurrent deviation from fair value : {df['deviation'].iloc[-1]:.4f}")
print(f"Current threshold (±)             : {(SPREAD_MULTIPLIER * df['volatility'].iloc[-1]):.4f}")
df[["close", "sma", "upper_band", "lower_band", "deviation"]].tail(10)

## 5. Feature Engineering

Derives ML-ready features from the indicators computed in section 4, and computes forward return labels.
**Must be run after section 4.** Adjust `FORWARD_BARS` and `PROFIT_THRESHOLD` to control label definition.

In [ ]:
# --- Label parameters: adjust these ---
FORWARD_BARS      = 5      # how many bars ahead to measure the return
PROFIT_THRESHOLD  = 0.0005 # minimum return to label a bar as profitable

# --- Features (derived from indicators computed in section 4) ---

# Normalized deviation: deviation scaled by volatility so it's comparable across vol regimes
df["deviation_norm"] = df["deviation"] / df["volatility"]

# SMA slope: how fast fair value is moving (rising or falling)
df["sma_slope"] = df["sma"].diff(5) / df["sma"].shift(5)

# Volume ratio: current volume relative to recent average (is this bar unusually active?)
df["volume_ratio"] = df["volume"] / df["volume"].rolling(20).mean()

# --- Label ---
df["forward_return"] = df["close"].shift(-FORWARD_BARS) / df["close"] - 1
df["label"] = (df["forward_return"] > PROFIT_THRESHOLD).astype(int)

FEATURES = ["deviation_norm", "volatility", "sma_slope", "volume_ratio"]
df_ml = df[FEATURES + ["label"]].dropna()

print(f"Forward bars: {FORWARD_BARS} | Profit threshold: {PROFIT_THRESHOLD}")
print(f"ML-ready rows: {len(df_ml)}")
print(f"Label distribution:\n{df_ml['label'].value_counts()}")
df_ml.head()

## 6. ML — Random Forest Signal Filter

Trains a binary classifier to predict whether a signal will be profitable.
Answers: *given these indicator conditions, is this signal worth acting on?*

Check `feature_importances_` after training to see which features the model found useful.
Poor recall on class `1` usually means the dataset is too small — get more data before drawing conclusions.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

X = df_ml[FEATURES]
y = df_ml["label"]

# shuffle=False is critical — test set must always be in the future relative to training set
X_train, X_test, y_train, y_test = train_test_split(X, y, shuffle=False)

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)
print(classification_report(y_test, model.predict(X_test)))

# Feature importance — which indicators does the model rely on most?
importance = pd.Series(model.feature_importances_, index=FEATURES).sort_values(ascending=False)
print("\nFeature importances:")
print(importance.round(4))

## 7. Parameter Optimization

Systematically tests combinations of `WINDOW` and `SPREAD_MULTIPLIER` and ranks them by PnL.
Answers: *what parameter settings produce the most profitable signals historically?*

This is separate from ML — it finds the best settings. ML then filters individual signals at those settings.
Run this on **in-sample data only**. Use the out-of-sample validation cell to verify the results generalize.

In [ ]:
import itertools

# --- Parameter grid: adjust ranges to explore ---
WINDOWS            = [5, 10, 20, 30]
SPREAD_MULTIPLIERS = [0.5, 1.0, 1.5, 2.0]

opt_results = []

for window, multiplier in itertools.product(WINDOWS, SPREAD_MULTIPLIERS):

    sma       = df["close"].rolling(window).mean()
    vol       = df["close"].rolling(window).std()
    deviation = df["close"] - sma

    position = 0
    trades   = []

    for i in range(len(df)):
        if pd.isna(sma.iloc[i]):
            continue
        threshold = multiplier * vol.iloc[i]
        dev       = deviation.iloc[i]
        price     = df["close"].iloc[i]

        if dev < -threshold and position < MAX_POSITION:
            trades.append({"action": "BUY",  "price": price})
            position += ORDER_SIZE
        elif dev > threshold and position > 0:
            trades.append({"action": "SELL", "price": price})
            position -= ORDER_SIZE

    buys  = [t["price"] for t in trades if t["action"] == "BUY"]
    sells = [t["price"] for t in trades if t["action"] == "SELL"]
    pairs = min(len(buys), len(sells))
    pnl   = sum(sells[i] - buys[i] for i in range(pairs))

    opt_results.append({
        "window":            window,
        "spread_multiplier": multiplier,
        "num_trades":        len(trades),
        "pnl":               round(pnl, 2),
    })

opt_df = pd.DataFrame(opt_results).sort_values("pnl", ascending=False)
print("Parameter optimization results (ranked by PnL):")
print(opt_df.to_string(index=False))

## 8. Lock In Parameters

Once satisfied with the optimization and ML results, run this cell to capture the chosen parameters.
The signal overlay and out-of-sample validation cells both read from `best_params`.

Update `WINDOW` and `SPREAD_MULTIPLIER` in section 4 to match your chosen values before running this cell.

In [ ]:
best_params = {
    "window":            WINDOW,
    "spread_multiplier": SPREAD_MULTIPLIER,
    "max_position":      MAX_POSITION,
    "order_size":        ORDER_SIZE,
}

print("Parameters locked in:")
for k, v in best_params.items():
    print(f"  {k}: {v}")

## 9. Signal Overlay
Visualize where the mean reversion strategy would have generated BUY and SELL signals.
Uses parameters from section 8 (best_params), falling back to section 4 values if not yet run.

In [ ]:
# Read parameters from best_params if available, else fall back to section 4 values
_window            = best_params.get("window",            WINDOW)
_spread_multiplier = best_params.get("spread_multiplier", SPREAD_MULTIPLIER)
_max_position      = best_params.get("max_position",      MAX_POSITION)
_order_size        = best_params.get("order_size",        ORDER_SIZE)

# Recompute indicators with locked-in params
df["sma"]        = df["close"].rolling(_window).mean()
df["volatility"] = df["close"].rolling(_window).std()
df["upper_band"] = df["sma"] + _spread_multiplier * df["volatility"]
df["lower_band"] = df["sma"] - _spread_multiplier * df["volatility"]
df["deviation"]  = df["close"] - df["sma"]

# Derive signals
df["signal"] = None
position = 0

for i in range(len(df)):
    row = df.iloc[i]
    if pd.isna(row["sma"]):
        continue
    threshold = _spread_multiplier * row["volatility"]
    deviation = row["deviation"]
    if deviation < -threshold and position < _max_position:
        df.at[df.index[i], "signal"] = "BUY"
        position += _order_size
    elif deviation > threshold and position > 0:
        df.at[df.index[i], "signal"] = "SELL"
        position = max(0, position - _order_size)

buy_signals  = df[df["signal"] == "BUY"]
sell_signals = df[df["signal"] == "SELL"]

print(f"Window={_window} | Multiplier={_spread_multiplier} | Max position={_max_position} | Order size={_order_size}")
print(f"BUY signals : {len(buy_signals)}")
print(f"SELL signals: {len(sell_signals)}")

x = np.arange(len(df))
buy_x  = [df.index.get_loc(i) for i in buy_signals.index]
sell_x = [df.index.get_loc(i) for i in sell_signals.index]
tick_pos, tick_labels = make_x_ticks(df.index)

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(x, df["close"],      color="#1f77b4", linewidth=1.0, label="Close",                    zorder=2)
ax.plot(x, df["sma"],        color="#ff7f0e", linewidth=1.0, label=f"SMA({_window})",          zorder=2)
ax.plot(x, df["upper_band"], color="#2ca02c", linewidth=0.8, linestyle="--", label="Upper band", zorder=2)
ax.plot(x, df["lower_band"], color="#d62728", linewidth=0.8, linestyle="--", label="Lower band", zorder=2)
ax.fill_between(x, df["lower_band"], df["upper_band"], alpha=0.06, color="gray")
ax.scatter(buy_x,  buy_signals["close"],  marker="^", color="#2ca02c", s=60, zorder=3, label=f"BUY ({len(buy_signals)})")
ax.scatter(sell_x, sell_signals["close"], marker="v", color="#d62728", s=60, zorder=3, label=f"SELL ({len(sell_signals)})")
ax.set_title(f"{SYMBOL} ({BAR_SIZE}) — Signal Overlay | window={_window}, multiplier={_spread_multiplier}, max_pos={_max_position}, order_size={_order_size}")
ax.set_ylabel("Price (USD)")
ax.legend(loc="upper left")
ax.grid(True, alpha=0.3)
ax.set_xticks(tick_pos)
ax.set_xticklabels(tick_labels, rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 10. Out-of-Sample Validation

Validates that the parameters found during research generalize to unseen data.

The fetched data (`df`) is split chronologically:
- **In-sample** — the first `IN_SAMPLE_FRAC` of bars (used for optimization and ML above)
- **Out-of-sample** — the remaining bars (never touched during research)

The parameter optimization is re-run on the out-of-sample period using only the best parameters
found in-sample. If PnL and trade count hold up, the parameters are likely to generalize.
If they collapse, the in-sample results were overfitting to that specific period.

**Never adjust parameters based on out-of-sample results.** It exists for final validation only.

In [ ]:
# --- Split ---
IN_SAMPLE_FRAC = 0.7  # 70% in-sample, 30% out-of-sample

split_idx  = int(len(df) * IN_SAMPLE_FRAC)
df_in      = df.iloc[:split_idx].copy()
df_out     = df.iloc[split_idx:].copy()

print(f"In-sample  : {len(df_in)} bars  ({df_in.index[0].date()} → {df_in.index[-1].date()})")
print(f"Out-of-sample: {len(df_out)} bars ({df_out.index[0].date()} → {df_out.index[-1].date()})")

# --- Re-run best params on out-of-sample data ---
_w = best_params["window"]
_m = best_params["spread_multiplier"]
_max_pos   = best_params["max_position"]
_order_sz  = best_params["order_size"]

sma_oos = df_out["close"].rolling(_w).mean()
vol_oos = df_out["close"].rolling(_w).std()
dev_oos = df_out["close"] - sma_oos

position  = 0
oos_trades = []

for i in range(len(df_out)):
    if pd.isna(sma_oos.iloc[i]):
        continue
    threshold = _m * vol_oos.iloc[i]
    dev       = dev_oos.iloc[i]
    price     = df_out["close"].iloc[i]
    if dev < -threshold and position < _max_pos:
        oos_trades.append({"action": "BUY",  "price": price})
        position += _order_sz
    elif dev > threshold and position > 0:
        oos_trades.append({"action": "SELL", "price": price})
        position -= _order_sz

oos_buys  = [t["price"] for t in oos_trades if t["action"] == "BUY"]
oos_sells = [t["price"] for t in oos_trades if t["action"] == "SELL"]
oos_pairs = min(len(oos_buys), len(oos_sells))
oos_pnl   = sum(oos_sells[i] - oos_buys[i] for i in range(oos_pairs))

# --- Compare in-sample vs out-of-sample ---
best_row = opt_df.iloc[0]
print(f"\nIn-sample best  — window={best_row['window']}, multiplier={best_row['spread_multiplier']}, trades={int(best_row['num_trades'])}, PnL={best_row['pnl']}")
print(f"Out-of-sample   — window={_w}, multiplier={_m}, trades={len(oos_trades)}, PnL={round(oos_pnl, 2)}")
print()
if oos_pnl > 0:
    print("OUT-OF-SAMPLE: PASS — PnL is positive. Parameters may generalize.")
else:
    print("OUT-OF-SAMPLE: FAIL — PnL is negative. Parameters may be overfitting in-sample period.")

## 11. Multi-Symbol Comparison
Compare relative performance and correlation across a basket of symbols.
Adjust `SYMBOLS` and `DURATION` freely.

In [ ]:
SYMBOLS  = ["SPY", "QQQ", "IWM"]
DURATION = "3 M"

closes = session.market_data.get_close_prices(SYMBOLS, duration=DURATION)

print(f"Fetched {len(closes)} aligned dates across {SYMBOLS}")
closes.tail()

In [ ]:
# Normalise to 100 at the start so all series are comparable on the same axis
normalised = (closes / closes.iloc[0]) * 100

fig, ax = plt.subplots(figsize=(14, 5))

for symbol in SYMBOLS:
    ax.plot(normalised.index, normalised[symbol], linewidth=1.5, label=symbol)

ax.axhline(100, color="gray", linewidth=0.8, linestyle="--")
ax.set_title(f"Relative Performance — Normalised to 100 ({DURATION})")
ax.set_ylabel("Indexed Price")
ax.legend()
ax.grid(True, alpha=0.3)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
ax.xaxis.set_major_locator(mdates.WeekdayLocator(interval=2))
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# Correlation matrix of daily returns
returns = closes.pct_change().dropna()
corr    = returns.corr()

print("Daily return correlation matrix:")
print(corr.round(4))

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(corr, vmin=-1, vmax=1, cmap="RdYlGn")
plt.colorbar(im, ax=ax)

ax.set_xticks(range(len(SYMBOLS)))
ax.set_yticks(range(len(SYMBOLS)))
ax.set_xticklabels(SYMBOLS)
ax.set_yticklabels(SYMBOLS)

for i in range(len(SYMBOLS)):
    for j in range(len(SYMBOLS)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=10)

ax.set_title("Return Correlation")
plt.tight_layout()
plt.show()

## 12. Teardown
Disconnect from TWS when you are done. Safe to skip if you plan to keep the session open.

In [ ]:
session.disconnect()
print("Disconnected. Session active:", session.is_connected)